# E053 · 56_data_imgprep

**데이터 처리** · `imgprep.py` — Pillow 이미지 전처리 → float32 묶음

**무엇을 보나**: 코드를 고치지 않은 프로그램을 일반 실행으로 한 번 돌려 최대 메모리 P를 잰 뒤, `memopro run --budget`으로 상한 P/2와 3P/4에서 다시 돌립니다.
- 결과(해시)가 일반 실행과 같아야 합니다.
- 최대 메모리가 상한 + 32MiB 안이거나, 지킬 수 없으면 **실행 중 경고와 초과량 보고**가 있어야 합니다(0.1.4의 약속, 0265).
- 시간 배율은 보고만 합니다.

**실행 방법**: Colab에서 이 노트북을 연 뒤 *런타임 → 런타임 유형 변경 → T4 GPU*를 고르고 *런타임 → 모두 실행*을 누르세요. 캐시가 없는 새 런타임을 가정합니다: memopro는 PyPI에서, 모델은 Hugging Face에서 새로 받습니다(토큰 불필요). 각 경우는 따로 된 프로세스에서 돌고, 끝에 경우별 표와 판정표가 출력됩니다. 결과 JSON은 `/content/memopro_bench/results/`에 남고, `SAVE_TO_DRIVE = True`면 Drive에도 복사됩니다.

연구 기록: docs/research/0268 (E053).

In [ ]:
# ==== 설정 (필요하면 여기만 고치세요) ====
MEMOPRO = "0.1.7"  # PyPI에서 설치할 memopro 버전
SAVE_TO_DRIVE = False  # True면 결과를 Google Drive(memopro_e053/)에도 복사합니다
SPEC = {'name': '56_data_imgprep', 'workload': 'imgprep.py', 'args': [], 'pip': [], 'timeout_s': 1800}
import os

CODE = os.environ.get("MP_BENCH_CODE", "/content/memopro_bench/code")
os.environ["MP_SAVE_TO_DRIVE"] = "1" if SAVE_TO_DRIVE else "0"

In [ ]:
# 이 노트북이 쓰는 코드를 새 런타임에 파일로 씁니다 (bench.py, worker.py, 작업 스크립트)
import os
FILES = {"bench.py": "\"\"\"E053 notebook helpers (docs/research/0268): set up a fresh Colab runtime, download models,\nrun each case in its own process, and print the verdicts. Written into /content by the notebook.\n\nNothing is assumed to be there before: memopro comes from PyPI, models from the Hugging Face\nHub (public repositories, no token), results go to /content/memopro_bench/results and, when\nSAVE_TO_DRIVE is on, to Google Drive as well.\n\"\"\"\n\nimport datetime\nimport glob\nimport json\nimport os\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport time\n\nROOT = os.environ.get(\"MP_BENCH_ROOT\", \"/content/memopro_bench\")\nHERE = os.path.dirname(os.path.abspath(__file__))\nMODELS = os.path.join(ROOT, \"models\")\nRESULTS = os.path.join(ROOT, \"results\")\nGiB, MiB = 1 << 30, 1 << 20\nTOLERANCE = 32 * MiB  # what memopro.enable allows before calling the ceiling passed (0265)\nWEIGHT_FILES = [\"*.json\", \"*.model\", \"*.txt\", \"*.tiktoken\", \"tokenizer*\"]\n\n\ndef log(msg):\n    print(f\"[{datetime.datetime.now().astimezone():%H:%M:%S}] {msg}\", flush=True)\n\n\ndef pip(*args):\n    r = subprocess.run(\n        [sys.executable, \"-m\", \"pip\", \"install\", \"-q\", *args],\n        capture_output=True,\n        text=True,\n        check=False,\n    )\n    if r.returncode:\n        log(f\"pip install {' '.join(args)} failed:\\n{r.stderr[-1500:]}\")\n    return r.returncode == 0\n\n\ndef setup(memopro_spec, extra=()):\n    \"\"\"Install memopro (and extras) from PyPI into this fresh runtime; describe the machine.\"\"\"\n    os.makedirs(RESULTS, exist_ok=True)\n    if os.environ.get(\"MP_BENCH_NO_INSTALL\") == \"1\":  # a local check of the notebook (build.py)\n        log(\"MP_BENCH_NO_INSTALL=1: using the installed packages\")\n    else:\n        log(f\"installing {memopro_spec} {' '.join(extra)}\")\n        pip(memopro_spec, *extra)\n    try:  # Colab's torchao < 0.16 makes peft refuse LoRA layers (0200)\n        from importlib.metadata import version\n\n        if tuple(int(x) for x in version(\"torchao\").split(\".\")[:2]) < (0, 16):\n            subprocess.run(\n                [sys.executable, \"-m\", \"pip\", \"uninstall\", \"-y\", \"-q\", \"torchao\"],\n                capture_output=True,\n                check=False,\n            )\n    except Exception as e:  # noqa: BLE001 - not installed\n        log(f\"torchao check skipped: {e}\")\n    env = {\n        \"time\": datetime.datetime.now().astimezone().isoformat(timespec=\"seconds\"),\n        \"python\": sys.version.split()[0],\n    }\n    for mod in (\"memopro\", \"torch\", \"transformers\", \"peft\", \"numpy\"):\n        r = subprocess.run(\n            [sys.executable, \"-c\", f\"import {mod}; print({mod}.__version__)\"],\n            capture_output=True,\n            text=True,\n            check=False,\n        )\n        env[mod] = r.stdout.strip() or \"missing\"\n    env[\"gpu\"] = \"none\"\n    if shutil.which(\"nvidia-smi\"):\n        smi = subprocess.run(\n            [\"nvidia-smi\", \"--query-gpu=name,memory.total\", \"--format=csv,noheader\"],\n            capture_output=True,\n            text=True,\n            check=False,\n        )\n        env[\"gpu\"] = smi.stdout.strip() or \"none\"\n    if env[\"gpu\"] == \"none\":\n        log(\"no GPU: choose Runtime > Change runtime type > T4 GPU (the data tests run on CPU)\")\n    env[\"host_memory_bytes\"] = os.sysconf(\"SC_PAGE_SIZE\") * os.sysconf(\"SC_PHYS_PAGES\")\n    env[\"disk_free_bytes\"] = shutil.disk_usage(\n        \"/content\" if os.path.isdir(\"/content\") else \"/\"\n    ).free\n    log(\n        f\"memopro {env['memopro']}, torch {env['torch']}, transformers {env['transformers']}; \"\n        f\"GPU {env['gpu']}; host {env['host_memory_bytes'] / GiB:.1f} GiB\"\n    )\n    return env\n\n\ndef fetch(repo, patterns):\n    \"\"\"Download `repo`'s files matching `patterns` (plus configs and tokenizer) into a folder of\n    real files (no symlinked cache); returns the folder and its weight bytes.\"\"\"\n    from huggingface_hub import snapshot_download\n\n    path = os.path.join(MODELS, repo.replace(\"/\", \"--\"))\n    t = time.time()\n    snapshot_download(repo, local_dir=path, allow_patterns=[*patterns, *WEIGHT_FILES])\n    index_safetensors(path)\n    size = sum(os.path.getsize(f) for f in glob.glob(os.path.join(path, \"*.safetensors\")))\n    log(f\"{repo}: {size / GiB:.1f} GiB of safetensors in {time.time() - t:.0f} s\")\n    return path, size\n\n\ndef index_safetensors(path):\n    \"\"\"Sharded safetensors without `model.safetensors.index.json` (CLIP-bigG ships only the .bin\n    index): write the index from the shards' headers so `from_pretrained` finds them (0270).\"\"\"\n    shards = sorted(glob.glob(os.path.join(path, \"*.safetensors\")))\n    index = os.path.join(path, \"model.safetensors.index.json\")\n    if len(shards) < 2 or glob.glob(os.path.join(path, \"*.safetensors.index.json\")):\n        return\n    weight_map = {}\n    for shard in shards:\n        with open(shard, \"rb\") as f:\n            header = json.loads(f.read(int.from_bytes(f.read(8), \"little\")))\n        weight_map.update({k: os.path.basename(shard) for k in header if k != \"__metadata__\"})\n    with open(index, \"w\") as f:\n        json.dump({\"metadata\": {}, \"weight_map\": weight_map}, f)\n    log(f\"wrote {os.path.basename(index)} for {len(shards)} shards\")\n\n\ndef drop(path):\n    shutil.rmtree(path, ignore_errors=True)\n\n\ndef run(name, cmd, timeout, env=None, cwd=None):\n    \"\"\"Run `cmd` in its own process group; the last JSON line it prints is the result. On a\n    timeout the whole group is killed.\"\"\"\n    log(f\"{name}: start\")\n    os.makedirs(RESULTS, exist_ok=True)\n    t = time.time()\n    p = subprocess.Popen(\n        cmd,\n        stdout=subprocess.PIPE,\n        stderr=subprocess.PIPE,\n        text=True,\n        env={**os.environ, **(env or {})},\n        cwd=cwd,\n        start_new_session=True,\n    )\n    try:\n        out, err = p.communicate(timeout=timeout)\n        rec = None\n        for line in reversed(out.strip().splitlines()):\n            if line.startswith(\"{\"):\n                try:\n                    rec = json.loads(line)\n                    break\n                except json.JSONDecodeError:\n                    pass\n        if rec is not None and \"status\" not in rec:  # a workload's own JSON line: it finished\n            rec[\"status\"] = \"ok\"\n        rec = rec or {\"status\": \"crashed\", \"returncode\": p.returncode, \"stderr\": err[-2500:]}\n    except subprocess.TimeoutExpired:\n        os.killpg(p.pid, signal.SIGKILL)\n        out, err = p.communicate()\n        rec = {\"status\": \"timeout\", \"timeout_s\": timeout}\n    rec.update(case=name, wall_s=round(time.time() - t, 1), warned_ceiling=\"cannot be kept\" in err)\n    if \"stderr\" not in rec and rec.get(\"status\") not in (\"ok\",):\n        rec[\"stderr\"] = err[-2500:]\n    with open(os.path.join(RESULTS, f\"{name}.json\"), \"w\") as f:\n        json.dump(rec, f, indent=1, default=str)\n    log(\n        f\"{name}: {rec.get('status')} in {rec['wall_s']} s\"\n        + (f\" — {rec.get('error', '')[:160]}\" if rec.get(\"status\") != \"ok\" else \"\")\n    )\n    return rec\n\n\ndef worker(name, args, timeout, python=None, env=None):\n    return run(\n        name,\n        [python or sys.executable, os.path.join(HERE, \"worker.py\"), *map(str, args)],\n        timeout,\n        env,\n    )\n\n\ndef script(name, path, args=(), timeout=3600, budget=None):\n    \"\"\"A data or library program, unchanged: plainly, or under `memopro run --budget`.\"\"\"\n    cmd = [sys.executable, path, *map(str, args)]\n    if budget is not None:\n        report = os.path.join(RESULTS, f\"{name}.report.json\")\n        cmd = [\n            sys.executable,\n            \"-m\",\n            \"memopro\",\n            \"run\",\n            \"--budget\",\n            str(budget),\n            \"--no-torch\",\n            \"--report-json\",\n            report,\n            path,\n            *map(str, args),\n        ]\n    rec = run(name, cmd, timeout, cwd=os.path.dirname(path))\n    if budget is not None:\n        rec[\"ceiling\"] = budget\n        if os.path.exists(report):\n            with open(report) as f:\n                session = json.load(f).get(\"session\", {})\n            rec[\"peak_footprint\"] = session.get(\"peak_footprint\")\n            rec[\"exceeded_bytes\"] = session.get(\"exceeded_bytes\")\n            rec[\"overruns\"] = session.get(\"overruns\")\n    return rec\n\n\ndef same(a, b, key):\n    return a.get(\"status\") == b.get(\"status\") == \"ok\" and a.get(key) == b.get(key)\n\n\ndef verdict(test, env, records, checks, notes=()):\n    \"\"\"Print the cases and the checks; save everything as one JSON (and to Drive if asked).\"\"\"\n    print(f\"\\n## {test}\\n\")\n    print(\"| case | status | wall s | host peak GiB | GPU peak GiB |\")\n    print(\"|---|---|---|---|---|\")\n    for r in records:\n        host = r.get(\"host_peak_bytes\") or r.get(\"peak_footprint\") or r.get(\"peak_bytes\")\n        gpu = r.get(\"gpu_peak_bytes\")\n        print(\n            f\"| {r['case']} | {r.get('status')} | {r.get('wall_s')} | \"\n            f\"{host / GiB if host else 0:.2f} | {gpu / GiB if gpu else 0:.2f} |\"\n        )\n    print(\"\\n| check | result | detail |\\n|---|---|---|\")\n    for label, ok, detail in checks:\n        mark = \"PASS\" if ok is True else \"FAIL\" if ok is False else \"report\"\n        print(f\"| {label} | {mark} | {detail} |\")\n    for n in notes:\n        print(f\"- {n}\")\n    out = {\n        \"test\": test,\n        \"env\": env,\n        \"records\": records,\n        \"checks\": [{\"check\": c, \"ok\": o, \"detail\": d} for c, o, d in checks],\n    }\n    path = os.path.join(RESULTS, f\"{test}.summary.json\")\n    with open(path, \"w\") as f:\n        json.dump(out, f, indent=1, default=str)\n    log(f\"saved {path}\")\n    if os.environ.get(\"MP_SAVE_TO_DRIVE\") == \"1\":\n        try:\n            from google.colab import drive\n\n            drive.mount(\"/content/drive\")\n            dest = os.path.join(\"/content/drive/MyDrive/memopro_e053\", test)\n            shutil.copytree(RESULTS, dest, dirs_exist_ok=True)\n            log(f\"copied the results to {dest}\")\n        except Exception as e:  # noqa: BLE001\n            log(f\"could not copy to Drive: {e}\")\n    return out\n", "worker.py": "\"\"\"E053 worker (docs/research/0268): one case in a fresh process; prints one JSON line.\n\n    worker.py lm_gen   <model_dir> <plain|memopro> <host_budget> <tokens> <prompts> [draft_dir]\n    worker.py lm_lora  <model_dir> <plain|memopro> <host_budget> <steps> <seq> <batch|auto>\n    worker.py vis_infer <model_dir> <class> <plain|memopro> <host_budget> <batch> <size>\n    worker.py vis_lora  <model_dir> <class> <plain|memopro> <host_budget> <steps> <batch> <size>\n    worker.py unsloth  <model_dir> <bits 16|4> <steps> <seq>   (run with Unsloth's own Python)\n\n`plain` loads the model normally onto the GPU; `memopro` streams it within `host_budget` bytes\nof host memory (memopro.rt.torch.stream_model / memopro.finetune). Self-contained: the notebook\nwrites this file into a fresh Colab runtime.\n\"\"\"\n\nimport gc\nimport hashlib\nimport json\nimport os\nimport sys\nimport threading\nimport time\nimport traceback\n\nif sys.argv[1] == \"unsloth\":\n    import unsloth  # noqa: F401  (before transformers and peft, as Unsloth asks)\nimport torch\n\nDEVICE = \"cuda\" if torch.cuda.is_available() else \"cpu\"\nRESULT = {\"task\": sys.argv[1], \"argv\": sys.argv[2:], \"device\": DEVICE, \"status\": None}\nPROMPTS = [\n    \"Explain how a hash table handles collisions.\",\n    \"Write a Python function that checks whether a string is a palindrome, with a short docstring.\",\n]\n\n\ndef sync():\n    if DEVICE == \"cuda\":\n        torch.cuda.synchronize()\n\n\ndef host_peak():\n    \"\"\"VmHWM: this process's own peak resident memory (ru_maxrss survives execve, 0199).\"\"\"\n    try:\n        with open(\"/proc/self/status\") as f:\n            for line in f:\n                if line.startswith(\"VmHWM:\"):\n                    return int(line.split()[1]) * 1024\n    except OSError:\n        pass\n    import resource\n\n    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    return r if sys.platform == \"darwin\" else r * 1024\n\n\nclass GpuMonitor:\n    \"\"\"Peak GPU memory in use as nvidia-smi sees it, sampled every 20 ms.\"\"\"\n\n    def __init__(self):\n        self.peak = 0\n        if DEVICE == \"cuda\":\n            threading.Thread(target=self._run, daemon=True).start()\n\n    def _run(self):\n        while True:\n            free, total = torch.cuda.mem_get_info()\n            self.peak = max(self.peak, total - free)\n            time.sleep(0.02)\n\n\nMON = GpuMonitor()\n\n\ndef finish(**kw):\n    RESULT.update(kw)\n    RESULT[\"status\"] = RESULT[\"status\"] or \"ok\"\n\n\ndef sha(t):\n    return hashlib.sha256(t.detach().float().cpu().numpy().tobytes()).hexdigest()[:16]\n\n\ndef stats_of(m):\n    rt = getattr(m, \"memopro_runtime\", None)\n    return rt.stats() if rt is not None else None\n\n\ndef wikitext(tok, n_tokens):\n    \"\"\"The first `n_tokens` tokens of WikiText-2's training split (the same text every case).\"\"\"\n    import datasets\n\n    ds = datasets.load_dataset(\"Salesforce/wikitext\", \"wikitext-2-raw-v1\", split=\"train\")\n    text = \"\".join(t for t in ds[\"text\"][:4000] if t.strip())\n    return tok(text, return_tensors=\"pt\").input_ids[0][:n_tokens]\n\n\ndef targets_of(m):\n    \"\"\"LoRA targets present in this architecture: separate q/k/v/o (Llama, Qwen, Mistral) or\n    fused qkv (Phi-3).\"\"\"\n    names = {n.split(\".\")[-1] for n, x in m.named_modules() if isinstance(x, torch.nn.Linear)}\n    for group in (\n        (\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\"),\n        (\"qkv_proj\", \"o_proj\"),\n        (\"query\", \"key\", \"value\"),\n        (\"q_proj\", \"v_proj\"),\n    ):\n        if all(g in names for g in group):\n            return list(group)\n    raise RuntimeError(f\"no known attention projections among {sorted(names)[:20]}\")\n\n\ndef load_lm(path, mode, budget):\n    import transformers\n\n    import memopro.rt.torch as rtt\n\n    t = time.time()\n    if mode == \"plain\":\n        m = transformers.AutoModelForCausalLM.from_pretrained(path, dtype=torch.bfloat16)\n        m = m.to(DEVICE).eval()\n    else:\n        m = rtt.stream_model(path, budget=budget, device=DEVICE)\n    sync()\n    RESULT[\"load_s\"] = round(time.time() - t, 1)\n    return m, transformers.AutoTokenizer.from_pretrained(path)\n\n\ndef lm_gen(path, mode, budget, new_tokens, prompts=None, draft=None):\n    import memopro\n    import memopro.rt.torch as rtt\n\n    m, tok = load_lm(path, mode, int(budget))\n    if draft:  # a resident draft proposes tokens; the text must not change (0276)\n        draft = rtt.draft_model(draft, target=m, device=DEVICE)\n        RESULT[\"draft_bytes\"] = draft.memopro_draft_bytes\n        RESULT[\"draft_dtype\"] = str(next(draft.parameters()).dtype)\n    tokens, secs = [], []\n    for p in PROMPTS[: int(prompts or len(PROMPTS))]:\n        msgs = [{\"role\": \"user\", \"content\": p}]\n        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors=\"pt\")\n        ids = (ids[\"input_ids\"] if hasattr(ids, \"keys\") else ids).to(DEVICE)\n        kw = {\n            \"max_new_tokens\": int(new_tokens),\n            \"min_new_tokens\": int(new_tokens),\n            \"pad_token_id\": tok.eos_token_id,\n        }\n        t = time.time()\n        if mode == \"plain\":\n            with torch.no_grad():\n                out = m.generate(input_ids=ids, do_sample=False, **kw)\n        else:\n            out = rtt.generate(m, ids, draft=draft or None, **kw)\n        sync()\n        secs.append(time.time() - t)\n        tokens.append(out[0, ids.shape[1] :].tolist())\n    finish(\n        tokens=tokens,\n        text=tok.decode(tokens[0], skip_special_tokens=True)[:200],\n        seconds=secs,\n        s_per_token=sum(secs) / (int(new_tokens) * len(secs)),\n        rt=stats_of(m),\n        copy_stats=getattr(getattr(m, \"memopro_weights\", None), \"copy_stats\", None),\n        memopro=memopro.__version__,\n    )\n\n\ndef lm_lora(path, mode, budget, steps, seq, batch):\n    import transformers\n\n    import memopro\n    import memopro.rt.torch as rtt\n\n    steps, seq = int(steps), int(seq)\n    tok = transformers.AutoTokenizer.from_pretrained(path)\n    ids = wikitext(tok, (seq - 1) * steps)\n    texts = [tok.decode(ids[i * (seq - 1) : (i + 1) * (seq - 1)]) for i in range(steps)]\n    # decoding and tokenizing again can add tokens (Mistral, 0270): cut into `steps` pieces exactly\n    eos = tok.eos_token or \"\"\n    seq = -(-(len(tok(eos.join(texts) + eos).input_ids) - 1) // steps)\n    if mode == \"memopro\":\n        m = rtt.stream_model(path, budget=int(budget), device=DEVICE)\n        targets = targets_of(m)  # before PEFT wraps them\n        r = memopro.finetune(\n            m,\n            texts,\n            tokenizer=tok,\n            seq_len=seq,\n            targets=tuple(targets),\n            batch=batch if batch == \"auto\" else int(batch),\n        )\n        finish(\n            losses=r.losses,\n            loss_bits=[float.hex(x) for x in r.losses],\n            batch=r.batch,\n            step_s=r.step_seconds,\n            tokens=r.tokens,\n            rt=stats_of(m),\n            targets=targets,\n        )\n        return\n    import peft\n\n    m = transformers.AutoModelForCausalLM.from_pretrained(path, dtype=torch.bfloat16).to(DEVICE)\n    all_ids = tok(eos.join(texts) + eos, return_tensors=\"pt\").input_ids[0]\n    torch.manual_seed(0)\n    targets = targets_of(m)\n    m = peft.get_peft_model(m, peft.LoraConfig(r=8, lora_alpha=16, target_modules=targets))\n    for p in m.parameters():\n        if p.requires_grad:\n            p.data = p.data.float()\n    m.train()\n    base = m.base_model.model\n    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\"use_reentrant\": True})\n    base.enable_input_require_grads()\n    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=2e-4)\n    losses, step_s = [], []\n    for i in range(0, len(all_ids) - 1, seq):\n        piece = all_ids[i : i + seq]\n        if len(piece) < 2:\n            continue\n        t = time.time()\n        loss = rtt.causal_lm_loss(base, piece.unsqueeze(0).to(DEVICE))\n        loss.backward()\n        opt.step()\n        opt.zero_grad(set_to_none=True)\n        sync()\n        losses.append(float(loss.detach()))\n        step_s.append(time.time() - t)\n    finish(losses=losses, loss_bits=[float.hex(x) for x in losses], step_s=step_s, targets=targets)\n\n\ndef vision(path, cls_name, mode, budget):\n    import transformers\n\n    import memopro.rt.torch as rtt\n\n    cls = getattr(transformers, cls_name)\n    t = time.time()\n    if mode == \"plain\":\n        m = cls.from_pretrained(path).to(DEVICE).eval()\n    else:\n        m = rtt.stream_model(path, budget=int(budget), device=DEVICE, model_class=cls)\n    sync()\n    RESULT[\"load_s\"] = round(time.time() - t, 1)\n    return m\n\n\ndef images(batch, size):\n    g = torch.Generator().manual_seed(0)\n    return torch.rand(int(batch), 3, int(size), int(size), generator=g)\n\n\ndef features(m, x):\n    out = m(pixel_values=x)\n    logits = getattr(out, \"logits\", None)\n    if logits is not None:\n        return logits\n    pooled = getattr(out, \"pooler_output\", None)\n    return pooled if pooled is not None else out.last_hidden_state[:, 0]\n\n\ndef vis_infer(path, cls_name, mode, budget, batch, size):\n    m = vision(path, cls_name, mode, budget)\n    x = images(batch, size).to(DEVICE)\n    with torch.no_grad():\n        t = time.time()\n        y = features(m, x)\n        sync()\n        first = time.time() - t\n        t = time.time()\n        y2 = features(m, x)\n        sync()\n    finish(\n        output_sha=sha(y),\n        same_twice=bool(torch.equal(y, y2)),\n        shape=list(y.shape),\n        first_s=first,\n        second_s=time.time() - t,\n        rt=stats_of(m),\n    )\n\n\ndef vis_lora(path, cls_name, mode, budget, steps, batch, size):\n    \"\"\"LoRA on the attention projections + a fresh 10-class head; deterministic kernels only\n    (strict, cuBLAS workspace fixed, attention on the math kernel: 0270).\"\"\"\n    import peft\n    from torch.nn.attention import SDPBackend, sdpa_kernel\n\n    import memopro.rt.torch as rtt\n\n    os.environ.setdefault(\"CUBLAS_WORKSPACE_CONFIG\", \":4096:8\")  # before cuBLAS starts\n    torch.use_deterministic_algorithms(True)\n    m = vision(path, cls_name, mode, budget)\n    torch.manual_seed(0)\n    m = peft.get_peft_model(m, peft.LoraConfig(r=8, lora_alpha=16, target_modules=targets_of(m)))\n    for layer in m.modules():  # adapters of a streamed model start on the meta device\n        if isinstance(layer, peft.tuners.lora.LoraLayer):\n            for name in layer.active_adapters:\n                if layer.lora_A[name].weight.device.type == \"meta\":\n                    layer.lora_A[name].to_empty(device=DEVICE)\n                    layer.lora_B[name].to_empty(device=DEVICE)\n                    layer.reset_lora_parameters(name, True)\n    for p in m.parameters():\n        if p.requires_grad:\n            p.data = p.data.float()\n    base = m.base_model.model\n    torch.manual_seed(1)\n    head = torch.nn.Linear(base.config.hidden_size, 10).to(DEVICE)\n    opt = torch.optim.AdamW(\n        [p for p in m.parameters() if p.requires_grad] + list(head.parameters()), lr=1e-3\n    )\n    x = images(batch, size).to(DEVICE)\n    y = torch.arange(int(batch), device=DEVICE) % 10\n    streamed = hasattr(base, \"memopro_weights\")\n    losses, step_s = [], []\n    for _ in range(int(steps)):\n        t = time.time()\n        with (\n            sdpa_kernel(SDPBackend.MATH),\n            rtt.saved_weights(base) if streamed else torch.enable_grad(),\n        ):\n            loss = torch.nn.functional.cross_entropy(head(features(base, x).float()), y)\n            loss.backward()\n        opt.step()\n        opt.zero_grad(set_to_none=True)\n        sync()\n        losses.append(float(loss.detach()))\n        step_s.append(time.time() - t)\n    finish(\n        losses=losses, loss_bits=[float.hex(v) for v in losses], step_s=step_s, rt=stats_of(base)\n    )\n\n\ndef unsloth_lora(path, bits, steps, seq):\n    import unsloth as us\n    from unsloth import FastLanguageModel\n\n    steps, seq = int(steps), int(seq)\n    RESULT[\"unsloth\"] = getattr(us, \"__version__\", \"?\")\n    t = time.time()\n    m, tok = FastLanguageModel.from_pretrained(\n        model_name=path, max_seq_length=seq, dtype=None, load_in_4bit=bits == \"4\"\n    )\n    RESULT[\"load_s\"] = round(time.time() - t, 1)\n    m = FastLanguageModel.get_peft_model(\n        m,\n        r=8,\n        lora_alpha=16,\n        lora_dropout=0,\n        bias=\"none\",\n        target_modules=[\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\"],\n        use_gradient_checkpointing=\"unsloth\",\n        random_state=0,\n    )\n    ids = wikitext(tok, (seq - 1) * steps)\n    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=2e-4)\n    losses, step_s = [], []\n    eos = torch.tensor([tok.eos_token_id])\n    for i in range(steps):\n        x = torch.cat([ids[i * (seq - 1) : (i + 1) * (seq - 1)], eos])[None].to(DEVICE)\n        t = time.time()\n        loss = m(input_ids=x, labels=x).loss\n        loss.backward()\n        opt.step()\n        opt.zero_grad(set_to_none=True)\n        sync()\n        losses.append(float(loss.detach()))\n        step_s.append(time.time() - t)\n    finish(losses=losses, step_s=step_s, dtype=str(m.dtype))\n\n\ndef main():\n    task, args = sys.argv[1], sys.argv[2:]\n    fn = {\n        \"lm_gen\": lm_gen,\n        \"lm_lora\": lm_lora,\n        \"vis_infer\": vis_infer,\n        \"vis_lora\": vis_lora,\n        \"unsloth\": unsloth_lora,\n    }[task]\n    try:\n        fn(*args)\n    except torch.cuda.OutOfMemoryError as e:\n        RESULT.update(status=\"oom\", error=str(e)[:400])\n    except MemoryError as e:\n        RESULT.update(status=\"oom\", error=repr(e)[:400])\n    except Exception as e:  # noqa: BLE001 - every failure is a recorded result\n        RESULT.update(\n            status=\"error\",\n            error=f\"{type(e).__name__}: {e}\"[:600],\n            traceback=traceback.format_exc()[-2500:],\n        )\n    gc.collect()\n    RESULT[\"host_peak_bytes\"] = host_peak()\n    RESULT[\"gpu_peak_bytes\"] = MON.peak\n    if DEVICE == \"cuda\":\n        RESULT[\"gpu_peak_allocated_bytes\"] = torch.cuda.max_memory_allocated()\n    print(\"\\n\" + json.dumps(RESULT, default=str), flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n", "workloads/common.py": "\"\"\"Shared by the E053 workloads (docs/research/0268): one JSON line at the end with the result\ndigests and this process's own peak memory (VmHWM; ru_maxrss survives execve, 0199).\"\"\"\n\nimport hashlib\nimport json\nimport sys\nimport time\n\nT0 = time.perf_counter()\n\n\ndef peak_bytes() -> int:\n    try:\n        with open(\"/proc/self/status\") as f:\n            for line in f:\n                if line.startswith(\"VmHWM:\"):\n                    return int(line.split()[1]) * 1024\n    except OSError:\n        pass\n    import resource\n\n    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    return r if sys.platform == \"darwin\" else r * 1024\n\n\ndef sha(*arrays) -> str:\n    h = hashlib.sha256()\n    for a in arrays:\n        h.update(memoryview(a).cast(\"B\"))\n    return h.hexdigest()[:16]\n\n\ndef done(**result) -> None:\n    result |= {\"seconds\": round(time.perf_counter() - T0, 2), \"peak_bytes\": peak_bytes()}\n    print(json.dumps(result), flush=True)\n", "workloads/imgprep.py": "\"\"\"E053 (docs/research/0268): image preprocessing as in a training pipeline — 1,500 synthetic\nRGB photos (768 x 768), each resized with Pillow to 512 x 512 and normalized into one float32\nbatch (4.7 GB), then per-channel statistics.\"\"\"\n\nimport numpy as np\nfrom common import done, sha\nfrom PIL import Image\n\nN, SRC, DST = 1500, 768, 512\nrng = np.random.default_rng(0)\nyy, xx = np.mgrid[0:SRC, 0:SRC].astype(np.float32) / SRC\nbatch = np.empty((N, DST, DST, 3), np.float32)\nmean = np.array([0.485, 0.456, 0.406], np.float32)\nstd = np.array([0.229, 0.224, 0.225], np.float32)\nfor i in range(N):\n    base = np.stack([np.sin(xx * (3 + i % 7)), np.cos(yy * (2 + i % 5)), xx * yy], -1)\n    img = ((base + 1) * 120 + rng.integers(0, 15, (SRC, SRC, 3))).clip(0, 255).astype(np.uint8)\n    small = np.asarray(Image.fromarray(img).resize((DST, DST), Image.BILINEAR), np.float32)\n    batch[i] = (small / 255 - mean) / std\ndone(\n    channel_mean=sha(batch.mean(axis=(0, 1, 2))),\n    channel_std=sha(batch.std(axis=(0, 1, 2))),\n    batch=sha(batch),\n)\n"}
for rel, text in FILES.items():
    p = os.path.join(CODE, rel)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, 'w') as f:
        f.write(text)
print("wrote", ", ".join(FILES))

In [ ]:
import sys, os
sys.path.insert(0, CODE)
import bench
env = bench.setup(f"memopro=={MEMOPRO}", SPEC["pip"])
wl = os.path.join(CODE, "workloads", SPEC["workload"])
SKIP = {"seconds", "peak_bytes", "case", "wall_s", "warned_ceiling", "status", "ceiling",
        "peak_footprint", "exceeded_bytes", "overruns", "stderr", "returncode", "timeout_s",
        "xgboost"}
result = lambda r: {k: v for k, v in r.items() if k not in SKIP}
plain = bench.script("plain", wl, SPEC["args"], SPEC["timeout_s"])
R, checks = [plain], []
if plain.get("status") != "crashed" and plain.get("peak_bytes"):
    P = plain["peak_bytes"]
    for frac in (0.5, 0.75):
        ceiling = int(P * frac)
        r = bench.script(f"memopro_{int(frac * 100)}pct", wl, SPEC["args"], 2 * SPEC["timeout_s"], budget=ceiling)
        R.append(r)
        done = "seconds" in r
        checks.append((f"결과 = 일반 실행 (상한 {frac:.0%})", done and result(r) == result(plain),
                       r.get("status") if not done else ""))
        peak, over = r.get("peak_footprint") or r.get("peak_bytes") or 0, r.get("exceeded_bytes") or 0
        if peak and peak <= ceiling + bench.TOLERANCE:
            c = (True, f"최대 {peak / bench.GiB:.2f} ≤ 상한 {ceiling / bench.GiB:.2f} GiB")
        elif r.get("warned_ceiling") and over:
            c = (True, f"상한을 지킬 수 없어 경고, +{over / bench.MiB:.0f} MiB 보고 (0265의 약속)")
        else:
            c = (False, f"최대 {peak / bench.GiB:.2f} GiB, 상한 {ceiling / bench.GiB:.2f}, 경고 {r.get('warned_ceiling')}")
        checks.append((f"상한 {frac:.0%}: 지킴 또는 경고·보고", *c))
        if done:
            checks.append((f"시간 배율 {frac:.0%}", None, f"{r['seconds'] / max(plain['seconds'], 1e-9):.2f}x ({r['seconds']:.0f}s / {plain['seconds']:.0f}s)"))
else:
    checks.append(("일반 실행", False, plain.get("status")))
bench.verdict(SPEC["name"], env, R, checks, [f"일반 실행 최대 메모리 {plain.get('peak_bytes', 0) / bench.GiB:.2f} GiB"])